<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/stage_06_00_model_training_plan.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06 – Model Training – Objetivo y Modelos**


# **Introducción**

Luego del proceso de selección de features, el problema ha sido transformado desde una serie de tiempo cruda hacia un dataset tabular compuesto por indicadores técnicos y variables contextuales. Estos features ya incorporan información temporal, de tendencia, momentum, reversión y volatilidad, reduciendo la necesidad de que el modelo aprenda directamente dependencias secuenciales complejas.

En este contexto, el problema deja de ser puramente secuencial y pasa a ser un problema de regresión tabular con señal débil y ruido elevado, típico en aplicaciones de mercados financieros intradía.

## **Objetivo**

El objetivo es entrenar modelos capaces de:

- capturar relaciones no lineales entre indicadores y el target
- ser robustos al ruido y a la baja señal
- generalizar correctamente en datos out-of-sample
- mantener estabilidad entre distintos períodos y regímenes de mercado

Dado este escenario, se priorizan modelos que han demostrado buen desempeño en problemas tabulares con estas características.


## **Justificación del uso de modelos tabulares**

La elección de modelos tabulares se basa en los siguientes puntos:

- los indicadores técnicos ya resumen la información temporal relevante
- no se observa una dependencia secuencial compleja que justifique modelos recurrentes o transformers
- los modelos tabulares son más robustos en presencia de señal débil
- permiten interpretar mejor la contribución de cada feature
- requieren menor complejidad computacional y de tuning

En consecuencia, se descarta el uso prioritario de modelos como LSTM o Transformers, ya que no aportan ventajas claras en este contexto.

## **Modelos a entrenar**

Se define el siguiente conjunto de modelos:

- Modelos lineales (baseline):
  - Ridge Regression

- Modelos de ensamble:
  - Random Forest

- Modelos de boosting (principales):
  - XGBoost
  - LightGBM

- Modelos no lineales simples (opcional):
  - MLP (Multi-Layer Perceptron)

## **Estrategia**














El enfoque de entrenamiento será progresivo:

1. Establecer un baseline con modelos lineales
2. Evaluar modelos de ensamble para capturar no linealidades
3. Priorizar modelos de boosting como candidatos principales
4. Comparar desempeño en métricas out-of-sample
5. Validar robustez y estabilidad de los resultados

Este enfoque permite identificar de forma clara si la complejidad adicional de los modelos se traduce en mejoras reales en capacidad predictiva.

# **0. Configuración del Entorno**


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [2]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [3]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

import os
import json
import logging
from pathlib import Path
from typing import Dict, Any, List, Tuple

import numpy as np
import pandas as pd

#from ta.momentum import ROCIndicator



In [4]:
# ============================================================
# Paths / IO (via env o defaults)
# ============================================================

DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

IN_PARQUET_DELTA_60 = Path(os.environ.get("IN_PARQUET_DELTA_60", "data/features/mnq_delta_60.parquet"))
IN_PARQUET_DELTA_90 = Path(os.environ.get("IN_PARQUET_DELTA_90", "data/features/mnq_delta_90.parquet"))

##############

OUT_SPLITS = Path(os.environ.get("OUT_PARQUET", "data/splits/splits.json"))


# PARA EL NOTEBOOK:

IN_PARQUET_DELTA_60 = DRIVE_DIR / IN_PARQUET_DELTA_60
IN_PARQUET_DELTA_90 = DRIVE_DIR / IN_PARQUET_DELTA_90






# **1. Carga de datos**

## **1.1. Carga de dataset `mnq_delta_*.parquet`**




In [5]:
def _ensure_parent_dir(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

def load_mnq_parquet(path: Path):
    os.path.exists(path)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(path)
    return mnq_parquet

## **1.2. Información de dataset `mnq_*`**



In [6]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")



## **1.3. Eliminación de OHLCV**

In [10]:
def delete_ohlcv(df, target_col):
    cols_to_drop = ["open", "high", "low", "close", "volume"]

    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    return df

## **1.3. Carga de mnq e información**




In [11]:
mnq_delta_60 = load_mnq_parquet(IN_PARQUET_DELTA_60)
mnq_delta_60 = delete_ohlcv(mnq_delta_60, "delta_60")
info_mnq_delta_60 = mnq_dataset_info(mnq_delta_60, name="mnq_delta_60", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_delta_60)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_delta_60
Shape: (700595, 9)
Columns: ['date', 'minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10', 'delta_60']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


In [12]:
mnq_delta_90 = load_mnq_parquet(IN_PARQUET_DELTA_90)
mnq_delta_90 = delete_ohlcv(mnq_delta_90, "delta_90")
info_mnq_delta_90 = mnq_dataset_info(mnq_delta_90, name="mnq_delta_90", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_delta_90)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_delta_90
Shape: (700595, 9)
Columns: ['date', 'minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10', 'delta_90']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2020-01-02 05:30:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2020-01-02  ->  2025-06-13
Total days (trading): 1295
Time-of-day range (minutes): {'min_minute_of_day': 330, 'max_minute_of_day': 870}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2020-01-02 10:30:00+00:00  ->  2025-06-13 18:30:00+00:00


In [13]:
mnq_delta_90

,date,minute_of_day,regime_id,ema_60,roc_30,roc_60,stoch_k_20,atr_norm_10,delta_90
datetime,,,,,,,,,
2020-01-02 05:30:00-05:00,2020-01-02,330,0,0.000410,0.039702,0.068079,90.476190,0.000115,-10.50
2020-01-02 05:31:00-05:00,2020-01-02,331,0,0.000342,0.034030,0.070922,80.952381,0.000118,-10.00
2020-01-02 05:32:00-05:00,2020-01-02,332,0,0.000358,0.017012,0.082277,85.714286,0.000115,-10.25
2020-01-02 05:33:00-05:00,2020-01-02,333,0,0.000264,0.025522,0.087963,71.428571,0.000112,-8.50
2020-01-02 05:34:00-05:00,2020-01-02,334,0,0.000310,0.031193,0.076600,80.952381,0.000112,-8.25
...,...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,866,3,-0.001801,-0.309818,-0.357839,30.739300,0.000671,-102.00
2025-06-13 14:27:00-04:00,2025-06-13,867,3,-0.002675,-0.406206,-0.419917,4.564315,0.000716,-74.75
2025-06-13 14:28:00-04:00,2025-06-13,868,3,-0.003444,-0.488852,-0.493419,10.233918,0.000782,-57.50


# **2. Separación de variables (X e y)**

En este punto del pipeline, el dataset ya contiene features diseñadas (indicadores técnicos y variables contextuales) y un target definido (`delta_60` o `delta_90`).

El objetivo ahora es **formalizar el problema de aprendizaje supervisado**, separando:

* **X (features):** variables explicativas que contienen la señal
* **y (target):** variable a predecir

Esta separación es necesaria porque los modelos de machine learning aprenden una función:

$$
f(X) \rightarrow y
$$

donde cada fila representa una observación independiente.

Es importante destacar que, aunque el dataset proviene de una serie temporal, en este enfoque tabular:

* la dependencia temporal ya está incorporada en los features (EMA, ROC, etc.)
* cada fila puede tratarse como una observación independiente
* la coherencia temporal se mantiene a través del índice datetime

Durante este paso, algunas columnas auxiliares (como `date`) se conservan en `X` **únicamente para facilitar el split temporal posterior**, pero no formarán parte del entrenamiento del modelo.


## **2.1. Código — Separación de X e y**

In [14]:
def separate_predictors_and_target(df, target_col):
    # asegurar orden temporal
    df = df.sort_index().copy()

    # target
    y = df[target_col].copy()

    # features (incluye columnas auxiliares como 'date')
    X = df.drop(columns=[target_col]).copy()

    # validación de alineación
    assert (X.index == y.index).all(), "X e y no están alineados"

    return X, y

In [15]:
X_60, y_60 = separate_predictors_and_target(
    mnq_delta_60,
    target_col="delta_60"
)

In [16]:
X_90, y_90 = separate_predictors_and_target(
    mnq_delta_90,
    target_col="delta_90"
)

**Resultado esperado**

* `X`: todas las columnas excepto el target (incluye `date` por ahora)
* `y`: serie con el target
* ambos perfectamente alineados por el índice datetime

# **3. Split temporal del dataset**

Una vez separados `X` e `y`, el siguiente paso es dividir el dataset en **train, valid y test**, respetando estrictamente el orden temporal.

A diferencia de problemas tradicionales, en datos financieros no se puede hacer un split aleatorio. La división debe realizarse por **días completos y en orden cronológico**, de modo que:

* el modelo se entrene solo con información pasada
* se evalúe sobre datos futuros (out-of-sample)
* se evite cualquier forma de *data leakage*

Para ello, se utilizan los días únicos del dataset y se asignan en proporción **70% / 15% / 15%** para train, valid y test, respectivamente.

Este enfoque garantiza una evaluación realista del modelo en condiciones similares a las de producción.


## **3.1. Código - Split temporal**

### **1. Validación de orden y alineación temporal**

El primero paso es comprobar la alineación temporal correcta:

In [18]:
def validate_X_y_alignment(X, y):
    if not X.index.is_monotonic_increasing:
        raise ValueError("X no está ordenado cronológicamente por índice.")

    if not y.index.is_monotonic_increasing:
        raise ValueError("y no está ordenado cronológicamente por índice.")

    if not X.index.equals(y.index):
        raise ValueError("X e y no tienen el mismo índice; no están alineados.")

    print("Validación OK: X e y están ordenados y alineados temporalmente.")
    print(f"Filas X: {len(X)} | Filas y: {len(y)}")
    print(f"Rango temporal: {X.index.min()} -> {X.index.max()}")

In [19]:
validate_X_y_alignment(X_60, y_60)

Validación OK: X e y están ordenados y alineados temporalmente.
Filas X: 700595 | Filas y: 700595
Rango temporal: 2020-01-02 05:30:00-05:00 -> 2025-06-13 14:30:00-04:00


In [20]:
validate_X_y_alignment(X_90, y_90)

Validación OK: X e y están ordenados y alineados temporalmente.
Filas X: 700595 | Filas y: 700595
Rango temporal: 2020-01-02 05:30:00-05:00 -> 2025-06-13 14:30:00-04:00


### **2. Obtención de días únicos**

In [21]:
def get_unique_trading_days(X, date_col="date"):
    if date_col not in X.columns:
        raise ValueError(f"No existe la columna '{date_col}' en X.")

    unique_days = pd.Index(pd.to_datetime(X[date_col]).dt.normalize().unique()).sort_values()

    print(f"Días únicos: {len(unique_days)}")
    print(f"Primer día: {unique_days.min().date()}")
    print(f"Último día: {unique_days.max().date()}")

    return unique_days

In [22]:
unique_days_60 = get_unique_trading_days(X_60, date_col="date")

Días únicos: 1295
Primer día: 2020-01-02
Último día: 2025-06-13


In [23]:
unique_days_90 = get_unique_trading_days(X_90, date_col="date")

Días únicos: 1295
Primer día: 2020-01-02
Último día: 2025-06-13


### **3. Dividir días train, valid y test**

In [24]:
def split_days_train_valid_test(unique_days, train_ratio=0.70, valid_ratio=0.15):
    n_days = len(unique_days)

    n_train = int(n_days * train_ratio)
    n_valid = int(n_days * valid_ratio)
    n_test = n_days - n_train - n_valid

    train_days = unique_days[:n_train]
    valid_days = unique_days[n_train:n_train + n_valid]
    test_days  = unique_days[n_train + n_valid:]

    print(f"Train days: {len(train_days)} | {train_days.min().date()} -> {train_days.max().date()}")
    print(f"Valid days: {len(valid_days)} | {valid_days.min().date()} -> {valid_days.max().date()}")
    print(f"Test days : {len(test_days)} | {test_days.min().date()} -> {test_days.max().date()}")

    return train_days, valid_days, test_days

In [25]:
train_days_60, valid_days_60, test_days_60 = split_days_train_valid_test(
    unique_days_60,
    train_ratio=0.70,
    valid_ratio=0.15,
)

Train days: 906 | 2020-01-02 -> 2023-10-30
Valid days: 194 | 2023-10-31 -> 2024-08-21
Test days : 195 | 2024-08-22 -> 2025-06-13


In [26]:
train_days_90, valid_days_90, test_days_90 = split_days_train_valid_test(
    unique_days_90,
    train_ratio=0.70,
    valid_ratio=0.15,
)

Train days: 906 | 2020-01-02 -> 2023-10-30
Valid days: 194 | 2023-10-31 -> 2024-08-21
Test days : 195 | 2024-08-22 -> 2025-06-13


### **4. Aplicar split sobre X e y**

In [27]:
def apply_day_split(X, y, train_days, valid_days, test_days, date_col="date"):
    x_days = pd.to_datetime(X[date_col]).dt.normalize()

    train_mask = x_days.isin(train_days)
    valid_mask = x_days.isin(valid_days)
    test_mask  = x_days.isin(test_days)

    X_train, y_train = X.loc[train_mask].copy(), y.loc[train_mask].copy()
    X_valid, y_valid = X.loc[valid_mask].copy(), y.loc[valid_mask].copy()
    X_test,  y_test  = X.loc[test_mask].copy(),  y.loc[test_mask].copy()

    print(f"Train -> X: {X_train.shape} | y: {y_train.shape}")
    print(f"Valid -> X: {X_valid.shape} | y: {y_valid.shape}")
    print(f"Test  -> X: {X_test.shape} | y: {y_test.shape}")

    return X_train, X_valid, X_test, y_train, y_valid, y_test

In [28]:
X_train_60, X_valid_60, X_test_60, y_train_60, y_valid_60, y_test_60 = apply_day_split(
    X_60,
    y_60,
    train_days_60,
    valid_days_60,
    test_days_60,
    date_col="date",
)

Train -> X: (490146, 8) | y: (490146,)
Valid -> X: (104954, 8) | y: (104954,)
Test  -> X: (105495, 8) | y: (105495,)


In [29]:
X_train_90, X_valid_90, X_test_90, y_train_90, y_valid_90, y_test_90 = apply_day_split(
    X_90,
    y_90,
    train_days_90,
    valid_days_90,
    test_days_90,
    date_col="date",
)

Train -> X: (490146, 8) | y: (490146,)
Valid -> X: (104954, 8) | y: (104954,)
Test  -> X: (105495, 8) | y: (105495,)


### **5. Validar no solapamiento**

In [30]:
def validate_no_day_overlap(train_days, valid_days, test_days):
    train_set = set(train_days)
    valid_set = set(valid_days)
    test_set = set(test_days)

    if train_set & valid_set:
        raise ValueError("Hay solapamiento entre train y valid.")
    if train_set & test_set:
        raise ValueError("Hay solapamiento entre train y test.")
    if valid_set & test_set:
        raise ValueError("Hay solapamiento entre valid y test.")

    print("Validación OK: no hay solapamiento entre días de train, valid y test.")

In [31]:
validate_no_day_overlap(train_days_60, valid_days_60, test_days_60)

Validación OK: no hay solapamiento entre días de train, valid y test.


In [32]:
validate_no_day_overlap(train_days_90, valid_days_90, test_days_90)

Validación OK: no hay solapamiento entre días de train, valid y test.


### **6. Eliminar columna 'date'**

In [34]:
def drop_date_column(X, cols_to_drop=None):
    if cols_to_drop is None:
        cols_to_drop = ["date"]

    X_clean = X.drop(columns=[c for c in cols_to_drop if c in X.columns]).copy()

    print(f"Shape original: {X.shape}")
    print(f"Shape limpia  : {X_clean.shape}")
    print(f"Columnas finales: {list(X_clean.columns)}")

    return X_clean

In [35]:
X_train_60 = drop_date_column(X_train_60)
X_valid_60 = drop_date_column(X_valid_60)
X_test_60  = drop_date_column(X_test_60)

Shape original: (490146, 8)
Shape limpia  : (490146, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Shape original: (104954, 8)
Shape limpia  : (104954, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Shape original: (105495, 8)
Shape limpia  : (105495, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']


In [36]:
X_train_90 = drop_date_column(X_train_90)
X_valid_90 = drop_date_column(X_valid_90)
X_test_90  = drop_date_column(X_test_90)

Shape original: (490146, 8)
Shape limpia  : (490146, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Shape original: (104954, 8)
Shape limpia  : (104954, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']
Shape original: (105495, 8)
Shape limpia  : (105495, 7)
Columnas finales: ['minute_of_day', 'regime_id', 'ema_60', 'roc_30', 'roc_60', 'stoch_k_20', 'atr_norm_10']


### **7. Guardado de splits**

#### **Verificación de orden**

In [37]:
def validate_and_sort_before_save(X, y, split_name="split"):
    if not X.index.equals(y.index):
        raise ValueError(f"{split_name}: X e y no están alineados por índice.")

    X = X.sort_index().copy()
    y = y.sort_index().copy()

    if not X.index.is_monotonic_increasing:
        raise ValueError(f"{split_name}: X no quedó ordenado cronológicamente.")
    if not y.index.is_monotonic_increasing:
        raise ValueError(f"{split_name}: y no quedó ordenado cronológicamente.")

    print(f"{split_name} OK | filas={len(X)} | {X.index.min()} -> {X.index.max()}")

    return X, y

In [38]:
X_train_60, y_train_60 = validate_and_sort_before_save(X_train_60, y_train_60, "train_60")
X_valid_60, y_valid_60 = validate_and_sort_before_save(X_valid_60, y_valid_60, "valid_60")
X_test_60,  y_test_60  = validate_and_sort_before_save(X_test_60,  y_test_60,  "test_60")

X_train_90, y_train_90 = validate_and_sort_before_save(X_train_90, y_train_90, "train_90")
X_valid_90, y_valid_90 = validate_and_sort_before_save(X_valid_90, y_valid_90, "valid_90")
X_test_90,  y_test_90  = validate_and_sort_before_save(X_test_90,  y_test_90,  "test_90")

train_60 OK | filas=490146 | 2020-01-02 05:30:00-05:00 -> 2023-10-30 14:30:00-04:00
valid_60 OK | filas=104954 | 2023-10-31 05:30:00-04:00 -> 2024-08-21 14:30:00-04:00
test_60 OK | filas=105495 | 2024-08-22 05:30:00-04:00 -> 2025-06-13 14:30:00-04:00
train_90 OK | filas=490146 | 2020-01-02 05:30:00-05:00 -> 2023-10-30 14:30:00-04:00
valid_90 OK | filas=104954 | 2023-10-31 05:30:00-04:00 -> 2024-08-21 14:30:00-04:00
test_90 OK | filas=105495 | 2024-08-22 05:30:00-04:00 -> 2025-06-13 14:30:00-04:00


#### **Definición de rutas**

In [39]:
from pathlib import Path
import os

##############

OUT_SPLITS = Path(os.environ.get("OUT_SPLITS", "data/splits/splits.json"))

# DELTA 60
OUT_PARQUET_DELTA_60_X_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_TRAIN", "data/splits/mnq_delta_60_X_train.parquet"))
OUT_PARQUET_DELTA_60_X_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_VALID", "data/splits/mnq_delta_60_X_valid.parquet"))
OUT_PARQUET_DELTA_60_X_TEST  = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_TEST",  "data/splits/mnq_delta_60_X_test.parquet"))

OUT_PARQUET_DELTA_60_Y_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_60_Y_TRAIN", "data/splits/mnq_delta_60_y_train.parquet"))
OUT_PARQUET_DELTA_60_Y_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_60_Y_VALID", "data/splits/mnq_delta_60_y_valid.parquet"))
OUT_PARQUET_DELTA_60_Y_TEST  = Path(os.environ.get("OUT_PARQUET_DELTA_60_Y_TEST",  "data/splits/mnq_delta_60_y_test.parquet"))

# DELTA 90
OUT_PARQUET_DELTA_90_X_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_TRAIN", "data/splits/mnq_delta_90_X_train.parquet"))
OUT_PARQUET_DELTA_90_X_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_VALID", "data/splits/mnq_delta_90_X_valid.parquet"))
OUT_PARQUET_DELTA_90_X_TEST  = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_TEST",  "data/splits/mnq_delta_90_X_test.parquet"))

OUT_PARQUET_DELTA_90_Y_TRAIN = Path(os.environ.get("OUT_PARQUET_DELTA_90_Y_TRAIN", "data/splits/mnq_delta_90_y_train.parquet"))
OUT_PARQUET_DELTA_90_Y_VALID = Path(os.environ.get("OUT_PARQUET_DELTA_90_Y_VALID", "data/splits/mnq_delta_90_y_valid.parquet"))
OUT_PARQUET_DELTA_90_Y_TEST  = Path(os.environ.get("OUT_PARQUET_DELTA_90_Y_TEST",  "data/splits/mnq_delta_90_y_test.parquet"))

# INPUTS
IN_PARQUET_DELTA_60 = DRIVE_DIR / IN_PARQUET_DELTA_60
IN_PARQUET_DELTA_90 = DRIVE_DIR / IN_PARQUET_DELTA_90

# OUTPUTS DELTA 60
OUT_PARQUET_DELTA_60_X_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_TRAIN
OUT_PARQUET_DELTA_60_X_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_VALID
OUT_PARQUET_DELTA_60_X_TEST  = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_TEST

OUT_PARQUET_DELTA_60_Y_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_60_Y_TRAIN
OUT_PARQUET_DELTA_60_Y_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_60_Y_VALID
OUT_PARQUET_DELTA_60_Y_TEST  = DRIVE_DIR / OUT_PARQUET_DELTA_60_Y_TEST

# OUTPUTS DELTA 90
OUT_PARQUET_DELTA_90_X_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_TRAIN
OUT_PARQUET_DELTA_90_X_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_VALID
OUT_PARQUET_DELTA_90_X_TEST  = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_TEST

OUT_PARQUET_DELTA_90_Y_TRAIN = DRIVE_DIR / OUT_PARQUET_DELTA_90_Y_TRAIN
OUT_PARQUET_DELTA_90_Y_VALID = DRIVE_DIR / OUT_PARQUET_DELTA_90_Y_VALID
OUT_PARQUET_DELTA_90_Y_TEST  = DRIVE_DIR / OUT_PARQUET_DELTA_90_Y_TEST

#### **Guardado de splits**

In [40]:
def save_splits_to_parquet(
    X_train, X_valid, X_test,
    y_train, y_valid, y_test,
    paths_dict
):
    # crear directorios si no existen
    for p in paths_dict.values():
        p.parent.mkdir(parents=True, exist_ok=True)

    # guardar X
    X_train.to_parquet(paths_dict["X_train"])
    X_valid.to_parquet(paths_dict["X_valid"])
    X_test.to_parquet(paths_dict["X_test"])

    # guardar y
    y_train.to_frame(name="target").to_parquet(paths_dict["y_train"])
    y_valid.to_frame(name="target").to_parquet(paths_dict["y_valid"])
    y_test.to_frame(name="target").to_parquet(paths_dict["y_test"])

    print("Guardado OK:")
    for k, v in paths_dict.items():
        print(f"{k}: {v}")

In [41]:
save_splits_to_parquet(
    X_train_60, X_valid_60, X_test_60,
    y_train_60, y_valid_60, y_test_60,
    {
        "X_train": OUT_PARQUET_DELTA_60_X_TRAIN,
        "X_valid": OUT_PARQUET_DELTA_60_X_VALID,
        "X_test":  OUT_PARQUET_DELTA_60_X_TEST,
        "y_train": OUT_PARQUET_DELTA_60_Y_TRAIN,
        "y_valid": OUT_PARQUET_DELTA_60_Y_VALID,
        "y_test":  OUT_PARQUET_DELTA_60_Y_TEST,
    }
)

Guardado OK:
X_train: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_X_train.parquet
X_valid: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_X_valid.parquet
X_test: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_X_test.parquet
y_train: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_y_train.parquet
y_valid: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_y_valid.parquet
y_test: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_60_y_test.parquet


In [42]:
save_splits_to_parquet(
    X_train_90, X_valid_90, X_test_90,
    y_train_90, y_valid_90, y_test_90,
    {
        "X_train": OUT_PARQUET_DELTA_90_X_TRAIN,
        "X_valid": OUT_PARQUET_DELTA_90_X_VALID,
        "X_test":  OUT_PARQUET_DELTA_90_X_TEST,
        "y_train": OUT_PARQUET_DELTA_90_Y_TRAIN,
        "y_valid": OUT_PARQUET_DELTA_90_Y_VALID,
        "y_test":  OUT_PARQUET_DELTA_90_Y_TEST,
    }
)

Guardado OK:
X_train: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_X_train.parquet
X_valid: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_X_valid.parquet
X_test: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_X_test.parquet
y_train: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_y_train.parquet
y_valid: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_y_valid.parquet
y_test: /content/drive/MyDrive/neural_profit/data/splits/mnq_delta_90_y_test.parquet


# **4. Escalado de features para modelos sensibles a la escala**

Una vez definidos y guardados los subconjuntos train, valid y test, el siguiente paso es preparar una versión escalada de las variables predictoras para aquellos modelos que dependen de la magnitud de los inputs.

El escalado no es necesario para todos los algoritmos. Modelos basados en árboles, como Random Forest, XGBoost o LightGBM, suelen ser invariantes a la escala de las features. En cambio, modelos lineales regularizados y redes neuronales, como Ridge y MLP, sí se benefician de trabajar con variables centradas y comparables en magnitud.

Para evitar data leakage, el escalador debe ajustarse exclusivamente con el conjunto de entrenamiento y luego aplicarse, sin recalibración, sobre valid y test. De esta forma, se preserva la causalidad temporal y se garantiza una evaluación fuera de muestra consistente.

## **4.1. Aplicación de escalador**

In [44]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

def fit_and_apply_standard_scaler(X_train, X_valid, X_test):
    scaler = StandardScaler()

    X_train_scaled = pd.DataFrame(
        scaler.fit_transform(X_train),
        index=X_train.index,
        columns=X_train.columns,
    )

    X_valid_scaled = pd.DataFrame(
        scaler.transform(X_valid),
        index=X_valid.index,
        columns=X_valid.columns,
    )

    X_test_scaled = pd.DataFrame(
        scaler.transform(X_test),
        index=X_test.index,
        columns=X_test.columns,
    )

    print("Escalado OK")
    print(f"Train: {X_train_scaled.shape}")
    print(f"Valid: {X_valid_scaled.shape}")
    print(f"Test : {X_test_scaled.shape}")

    return X_train_scaled, X_valid_scaled, X_test_scaled, scaler

In [47]:
X_train_60_scaled, X_valid_60_scaled, X_test_60_scaled, scaler_60 = fit_and_apply_standard_scaler(
    X_train_60,
    X_valid_60,
    X_test_60,
)

Escalado OK
Train: (490146, 7)
Valid: (104954, 7)
Test : (105495, 7)


In [48]:
X_train_90_scaled, X_valid_90_scaled, X_test_90_scaled, scaler_90 = fit_and_apply_standard_scaler(
    X_train_90,
    X_valid_90,
    X_test_90,
)

Escalado OK
Train: (490146, 7)
Valid: (104954, 7)
Test : (105495, 7)


## **4.2. Validación de escalamiento**

In [59]:
def validate_scaled_data(X_train_scaled, X_valid_scaled, X_test_scaled, name="dataset"):
    mean_train = X_train_scaled.mean()
    std_train = X_train_scaled.std()

    print(f"=== Validación {name} ===")
    print("Media (train) — debería ~ 0:")
    print(mean_train.round(4))

    print("\nStd (train) — debería ~ 1:")
    print(std_train.round(4))

    if not X_train_scaled.columns.equals(X_valid_scaled.columns) or not X_train_scaled.columns.equals(X_test_scaled.columns):
        raise ValueError("Columnas inconsistentes entre train/valid/test")

    print("\nColumnas consistentes")
    print(f"Shapes: train={X_train_scaled.shape}, valid={X_valid_scaled.shape}, test={X_test_scaled.shape}")

## **4.3. Guardado de escalamiento**

In [58]:
import joblib

def save_scaled_pipeline(
    X_train_scaled, X_valid_scaled, X_test_scaled,
    scaler,
    paths_dict,
    scaler_path
):
    # crear carpetas
    for p in list(paths_dict.values()) + [scaler_path]:
        p.parent.mkdir(parents=True, exist_ok=True)

    # guardar datasets
    X_train_scaled.to_parquet(paths_dict["X_train"])
    X_valid_scaled.to_parquet(paths_dict["X_valid"])
    X_test_scaled.to_parquet(paths_dict["X_test"])

    # guardar scaler
    joblib.dump(scaler, scaler_path)

    print("\n=== Guardado OK ===")
    for k, v in paths_dict.items():
        print(f"{k}: {v}")

    print(f"Scaler: {scaler_path}")

### **Definición de rutas**

In [56]:
from pathlib import Path
import os

# DELTA 60 ESCALADO
OUT_PARQUET_DELTA_60_X_TRAIN_SCALED = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_TRAIN_SCALED", "data/scaled/mnq_delta_60_X_train_scaled.parquet"))
OUT_PARQUET_DELTA_60_X_VALID_SCALED = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_VALID_SCALED", "data/scaled/mnq_delta_60_X_valid_scaled.parquet"))
OUT_PARQUET_DELTA_60_X_TEST_SCALED  = Path(os.environ.get("OUT_PARQUET_DELTA_60_X_TEST_SCALED",  "data/scaled/mnq_delta_60_X_test_scaled.parquet"))

# DELTA 90 ESCALADO
OUT_PARQUET_DELTA_90_X_TRAIN_SCALED = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_TRAIN_SCALED", "data/scaled/mnq_delta_90_X_train_scaled.parquet"))
OUT_PARQUET_DELTA_90_X_VALID_SCALED = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_VALID_SCALED", "data/scaled/mnq_delta_90_X_valid_scaled.parquet"))
OUT_PARQUET_DELTA_90_X_TEST_SCALED  = Path(os.environ.get("OUT_PARQUET_DELTA_90_X_TEST_SCALED",  "data/scaled/mnq_delta_90_X_test_scaled.parquet"))

OUT_PARQUET_DELTA_60_X_TRAIN_SCALED = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_TRAIN_SCALED
OUT_PARQUET_DELTA_60_X_VALID_SCALED = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_VALID_SCALED
OUT_PARQUET_DELTA_60_X_TEST_SCALED  = DRIVE_DIR / OUT_PARQUET_DELTA_60_X_TEST_SCALED

OUT_PARQUET_DELTA_90_X_TRAIN_SCALED = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_TRAIN_SCALED
OUT_PARQUET_DELTA_90_X_VALID_SCALED = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_VALID_SCALED
OUT_PARQUET_DELTA_90_X_TEST_SCALED  = DRIVE_DIR / OUT_PARQUET_DELTA_90_X_TEST_SCALED

OUT_SCALER_DELTA_60 = Path(os.environ.get("OUT_SCALER_DELTA_60", "data/scaled/scaler_delta_60.pkl"))
OUT_SCALER_DELTA_90 = Path(os.environ.get("OUT_SCALER_DELTA_90", "data/scaled/scaler_delta_90.pkl"))

OUT_SCALER_DELTA_60 = DRIVE_DIR / OUT_SCALER_DELTA_60
OUT_SCALER_DELTA_90 = DRIVE_DIR / OUT_SCALER_DELTA_90

## **4.4. Verificación y guardado final**

In [60]:
validate_scaled_data(X_train_90_scaled, X_valid_90_scaled, X_test_90_scaled, "delta_90")

save_scaled_pipeline(
    X_train_90_scaled, X_valid_90_scaled, X_test_90_scaled,
    scaler_90,
    {
        "X_train": OUT_PARQUET_DELTA_90_X_TRAIN_SCALED,
        "X_valid": OUT_PARQUET_DELTA_90_X_VALID_SCALED,
        "X_test":  OUT_PARQUET_DELTA_90_X_TEST_SCALED,
    },
    OUT_SCALER_DELTA_90
)

=== Validación delta_90 ===
Media (train) — debería ~ 0:
minute_of_day   -0.0
regime_id       -0.0
ema_60           0.0
roc_30           0.0
roc_60           0.0
stoch_k_20       0.0
atr_norm_10      0.0
dtype: float64

Std (train) — debería ~ 1:
minute_of_day    1.0
regime_id        1.0
ema_60           1.0
roc_30           1.0
roc_60           1.0
stoch_k_20       1.0
atr_norm_10      1.0
dtype: float64

Columnas consistentes
Shapes: train=(490146, 7), valid=(104954, 7), test=(105495, 7)

=== Guardado OK ===
X_train: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_90_X_train_scaled.parquet
X_valid: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_90_X_valid_scaled.parquet
X_test: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_90_X_test_scaled.parquet
Scaler: /content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_90.pkl


In [61]:
validate_scaled_data(X_train_60_scaled, X_valid_60_scaled, X_test_60_scaled, "delta_60")

save_scaled_pipeline(
    X_train_60_scaled, X_valid_60_scaled, X_test_60_scaled,
    scaler_60,
    {
        "X_train": OUT_PARQUET_DELTA_60_X_TRAIN_SCALED,
        "X_valid": OUT_PARQUET_DELTA_60_X_VALID_SCALED,
        "X_test":  OUT_PARQUET_DELTA_60_X_TEST_SCALED,
    },
    OUT_SCALER_DELTA_60
)

=== Validación delta_60 ===
Media (train) — debería ~ 0:
minute_of_day   -0.0
regime_id       -0.0
ema_60           0.0
roc_30           0.0
roc_60           0.0
stoch_k_20       0.0
atr_norm_10      0.0
dtype: float64

Std (train) — debería ~ 1:
minute_of_day    1.0
regime_id        1.0
ema_60           1.0
roc_30           1.0
roc_60           1.0
stoch_k_20       1.0
atr_norm_10      1.0
dtype: float64

Columnas consistentes
Shapes: train=(490146, 7), valid=(104954, 7), test=(105495, 7)

=== Guardado OK ===
X_train: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_60_X_train_scaled.parquet
X_valid: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_60_X_valid_scaled.parquet
X_test: /content/drive/MyDrive/neural_profit/data/scaled/mnq_delta_60_X_test_scaled.parquet
Scaler: /content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl


# **5. Predicción de modelos**

Los modelos a entrenar tienen como objetivo predecir una variable continua:

* `delta_60` o `delta_90`

Estas representan el **movimiento futuro del precio** en puntos a un horizonte fijo (60 o 90 minutos).

Por lo tanto, el problema se define como una **regresión supervisada**, donde:

* **Input (`X`)**: indicadores técnicos + variables contextuales
* **Output (`y`)**: retorno futuro (delta)

Cada modelo generará una predicción:

$$
\hat{y}_t = f(X_t)
$$

donde $\hat{y}_t$ es la estimación del movimiento futuro del mercado en ese instante.

---

**Sobre el uso de datos escalados**

No afecta la comparación, y esto es importante entenderlo bien:

* El escalado **solo transforma el espacio de entrada (`X`)**
* El target (`y`) **no se escala**
* Las predicciones siempre están en la **misma unidad original (puntos)**

Por lo tanto:

* MAE, RMSE, R² → comparables ✔️
* Directional accuracy → comparable ✔️

---

**Punto clave**

> Dos modelos pueden usar representaciones distintas de `X` (escalado vs no escalado),
> pero mientras predigan el mismo `y`, la comparación es completamente válida.

---

**Conclusión**

* No hay sesgo en comparar modelos escalados vs no escalados
* Es una práctica estándar en ML tabular
* Lo importante es que todos predicen el mismo target y se evalúan igual

## **5.1. Función unificada de métricas**

Dado que se entrenarán múltiples modelos sobre el mismo problema de regresión (`delta_60` y `delta_90`), resulta conveniente centralizar el proceso de evaluación en una única función estandarizada.

Esta función tendrá como propósito recibir los valores reales (`y_true`) y las predicciones del modelo (`y_pred`), calcular un conjunto consistente de métricas y devolver los resultados en un formato estructurado que facilite su comparación.

La unificación de este proceso permite:

* garantizar que todos los modelos sean evaluados bajo exactamente los mismos criterios
* evitar la duplicación de lógica en distintos notebooks o etapas del pipeline
* asegurar consistencia en la medición del desempeño, especialmente en datos fuera de muestra
* facilitar la trazabilidad y reproducibilidad de los resultados

De esta manera, se establece una base sólida y homogénea para la comparación objetiva entre modelos, independientemente de su complejidad o naturaleza.




In [62]:
"""
Métricas comunes para modelos de regresión tabular en el proyecto neural_profit.

Este módulo define funciones reutilizables para evaluar predicciones de modelos
sobre targets continuos como delta_60 y delta_90.

Uso típico desde notebook:
    from stage_07_metrics import evaluate_regression_predictions

    metrics = evaluate_regression_predictions(
        y_true=y_valid,
        y_pred=y_pred_valid,
        split_name="valid",
        model_name="ridge",
        target_name="delta_60",
    )
"""

from __future__ import annotations

from typing import Any, Dict, Optional

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def _to_1d_numpy(x: Any, name: str) -> np.ndarray:
    """
    Convierte una entrada tipo pandas/numpy/lista a un array 1D de numpy.
    """
    if isinstance(x, pd.Series):
        arr = x.to_numpy()
    elif isinstance(x, pd.DataFrame):
        if x.shape[1] != 1:
            raise ValueError(f"{name} DataFrame debe tener una sola columna.")
        arr = x.iloc[:, 0].to_numpy()
    else:
        arr = np.asarray(x)

    arr = np.ravel(arr)

    if arr.ndim != 1:
        raise ValueError(f"{name} no pudo convertirse a vector 1D.")

    return arr


def directional_accuracy(y_true: Any, y_pred: Any) -> float:
    """
    Calcula directional accuracy comparando el signo de y_true y y_pred.
    """
    y_true_arr = _to_1d_numpy(y_true, "y_true")
    y_pred_arr = _to_1d_numpy(y_pred, "y_pred")

    if len(y_true_arr) != len(y_pred_arr):
        raise ValueError("y_true y y_pred deben tener la misma longitud.")

    true_sign = np.sign(y_true_arr)
    pred_sign = np.sign(y_pred_arr)

    return float(np.mean(true_sign == pred_sign))


def evaluate_regression_predictions(
    y_true: Any,
    y_pred: Any,
    split_name: Optional[str] = None,
    model_name: Optional[str] = None,
    target_name: Optional[str] = None,
) -> Dict[str, Any]:
    """
    Evalúa predicciones de regresión con métricas comunes.

    Parámetros
    ----------
    y_true : array-like
        Valores reales.
    y_pred : array-like
        Predicciones del modelo.
    split_name : str, opcional
        Nombre del split, por ejemplo: train, valid, test.
    model_name : str, opcional
        Nombre del modelo, por ejemplo: ridge, xgboost.
    target_name : str, opcional
        Nombre del target, por ejemplo: delta_60.

    Retorna
    -------
    dict
        Diccionario con métricas de evaluación.
    """
    y_true_arr = _to_1d_numpy(y_true, "y_true")
    y_pred_arr = _to_1d_numpy(y_pred, "y_pred")

    if len(y_true_arr) != len(y_pred_arr):
        raise ValueError("y_true y y_pred deben tener la misma longitud.")

    if len(y_true_arr) == 0:
        raise ValueError("No se puede evaluar un vector vacío.")

    mae = mean_absolute_error(y_true_arr, y_pred_arr)
    rmse = np.sqrt(mean_squared_error(y_true_arr, y_pred_arr))
    r2 = r2_score(y_true_arr, y_pred_arr)
    da = directional_accuracy(y_true_arr, y_pred_arr)

    metrics = {
        "model": model_name,
        "target": target_name,
        "split": split_name,
        "n_samples": int(len(y_true_arr)),
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
        "directional_accuracy": float(da),
    }

    return metrics


def metrics_dict_to_frame(metrics: Dict[str, Any]) -> pd.DataFrame:
    """
    Convierte un diccionario de métricas en DataFrame de una fila.
    """
    return pd.DataFrame([metrics])


def print_metrics(metrics: Dict[str, Any], decimals: int = 6) -> None:
    """
    Imprime métricas de forma legible.
    """
    print("=== Regression Metrics ===")
    for key, value in metrics.items():
        if isinstance(value, float):
            print(f"{key}: {value:.{decimals}f}")
        else:
            print(f"{key}: {value}")

## **3.2. Qué NO es este problema**


Quedan explícitamente fuera de alcance en este stage:

- Predicción de trayectorias completas multi-step adicionales fuera del esquema formal definido.
- Targets agregados ad-hoc (suma, promedio o retornos acumulados recalculados fuera del target previamente definido).
- Entrenamiento de múltiples regresiones independientes por paso futuro sin estructura temporal compartida.
- Modificaciones del esquema `seq2one` o `seq2seq` fuera de la formulación establecida.

Este stage se limita exclusivamente a entrenar modelos bajo una formulación bien definida y controlada del problema.



# **4.  Protocolo experimental común de entrenamiento**

Se entrenan múltiples modelos bajo reglas estrictamente comunes, garantizando comparabilidad entre configuraciones:

- Mismo dataset.
- Mismo split temporal (train / valid / test).
- Mismo escalamiento.
- Misma definición del target.
- Misma función objetivo.

Ejemplos típicos (alineados al enfoque del libro):

- Naive (baseline)
- Modelos lineales (Ridge, Lasso)
- Árboles y ensembles
- Redes neuronales (MLP, LSTM, etc.)

Cada modelo:

- Se entrena utilizando **exclusivamente el set de entrenamiento**.
- Tiene hiperparámetros y regularización definidos previamente.
- No se realizan ajustes posteriores basados en métricas del set de test.

El objetivo de este protocolo es asegurar que cualquier diferencia observada en desempeño se deba al modelo y no a variaciones en el proceso experimental.


# **6. Definición de la comparación Predicción vs Valor Real**

En este stage, la evaluación del modelo se realiza bajo los esquemas **`seq2one`** y **`seq2seq`**.  
Cada ventana histórica produce:

- una predicción escalar (seq2one), o  
- una secuencia futura completa (seq2seq).

La comparación siempre se realiza directamente entre predicción y valor real correspondiente, respetando la formulación original del problema.



## **6.1 Esquema de predicción**

### **6.1.1. Enfoque `seq2one``**

Para cada muestra $t$, el modelo recibe como entrada:

$$
X_t \in \mathbb{R}^{L \times N}
$$

correspondiente a $L$ minutos consecutivos con $N$ features por minuto.

El modelo produce una única predicción escalar:

$$
\hat{y}_t \in \mathbb{R}
$$

que representa el valor futuro del target definido (por ejemplo, $\Delta_h$ para un horizonte fijo $h$).

Se compara directamente contra el valor real observado:

$$
y_t \in \mathbb{R}
$$

La comparación es **escalar contra escalar**, sin:

- generar trayectorias,
- aplicar agregaciones intermedias,
- ni comparar secuencias completas.

Cada ventana histórica tiene una única predicción y un único valor real asociado.


### **6.1.2. Enfoque `seq2seq`**

Para cada muestra $t$, el modelo recibe como entrada:

$$
X_t \in \mathbb{R}^{L \times N}
$$

A partir de esta entrada, el modelo predice una secuencia futura completa:

$$
\hat{Y}_{t+1:t+L} \in \mathbb{R}^{L \times 1}
$$

es decir, un valor del target por cada uno de los $L$ pasos futuros.

La predicción se compara directamente contra la secuencia real futura observada:

$$
Y^{real}_{t+1:t+L} \in \mathbb{R}^{L \times 1}
$$

La comparación es **secuencia contra secuencia**, sin colapsar el target ni aplicar agregaciones previas.


## **6.2 Cálculo de métricas**

### **6.2.1. Enfoque `seq2one``**

A partir de la comparación $\hat{y}_t$ vs $y_t$, las métricas se calculan
sobre el conjunto completo de muestras del split correspondiente.

Métricas utilizadas:

- MAE
- RMSE
- R² (opcional)
- Métricas direccionales (signo de la predicción vs signo real)

Las métricas se agregan sobre todas las ventanas del split.

### **6.2.2. Enfoque `seq2seq`**


Las métricas se calculan bajo dos perspectivas:

**(A) Por paso temporal**

Comparación entre:

$$
\hat{y}_{t+k} \quad \text{vs} \quad y^{real}_{t+k}, \quad k = 1, \dots, L
$$

**(B) Sobre la trayectoria completa**

Error global entre:

$$
\hat{Y}_{t+1:t+L} \quad \text{vs} \quad Y^{real}_{t+1:t+L}
$$

No se realiza comparación contra un escalar ni contra un valor agregado final.

El problema se mantiene estrictamente bajo la formulación `seq2seq`.

## **6.3. Uso de los splits (criterio de evaluación)**


Siguiendo el criterio operativo del workflow:

- **TRAIN**: utilizado exclusivamente para el aprendizaje del modelo.
- **VALID**: utilizado para medir desempeño y descartar modelos que no generalizan.
- **TEST**: utilizado únicamente una vez finalizada la selección del modelo.

No se ajustan hiperparámetros en función del set de test.

El modelo se entrena solo con TRAIN, pero se evalúa con VALID para decidir si es candidato a pasar al stage siguiente.

En términos operativos:

- TRAIN → para aprender.
- VALID → para medir desempeño y comparar modelos.
- TEST → solo al final, una vez elegido el mejor modelo.

Este esquema evita contaminación de información y garantiza evaluación fuera de muestra.


## **6.4. Derivación de métricas**


A partir de la comparación entre predicción y valor real (según el esquema `seq2one` o `seq2seq`), se derivan dos grupos de métricas:

- **(A) Métricas de Machine Learning**

  - MAE
  - RMSE
  - R² (opcional)
  - Métricas direccionales (signo de la predicción vs signo real)

  Estas métricas se calculan sobre VALID durante el stage_06 para decidir qué modelos continúan.

- **(B) Métricas económicas**

  Una vez seleccionados los modelos candidatos, se derivan métricas económicas utilizando el delta real observado dentro de la ventana futura:

  - Valor esperado (EV)
  - Ratio TP/SL
  - Drawdown
  - Métricas de riesgo-retorno

  Estas métricas permiten traducir el desempeño estadístico en impacto operativo.

La comparación económica definitiva se realiza posteriormente en el stage_07.

# **7. Métricas de predicción (Machine Learning)**

La evaluación del desempeño se realiza exclusivamente fuera de muestra (VALID).  
El conjunto TEST se reserva únicamente para la evaluación final una vez seleccionado el modelo.

Las métricas se organizan según:

- Métricas principales (criterio de selección)
- Métricas complementarias (interpretación)
- Métricas diagnósticas (solo análisis interno)

## **7.1. Enfoque `seq2one`**

Dado que el problema consiste en la predicción de un valor escalar futuro, la comparación se realiza entre:

$$
\hat{y}_t \quad \text{vs} \quad y_t
$$

### **7.1.1 Métricas principales (criterio de selección)**

**1. MAE (Mean Absolute Error)**

Error absoluto medio entre predicción y valor real:

$$
\text{MAE} = \frac{1}{N} \sum_{t=1}^{N} |\hat{y}_t - y_t|
$$

**2. RMSE (Root Mean Squared Error)**

Raíz del error cuadrático medio:

$$
\text{RMSE} = \sqrt{\frac{1}{N} \sum_{t=1}^{N} (\hat{y}_t - y_t)^2}
$$

Estas métricas constituyen el **criterio principal de comparación y ranking** de modelos.



### **7.1.2 Métricas complementarias**


**3. Directional Accuracy (DA)**

$$
DA = P\left[\text{sign}(\hat{y}_t) = \text{sign}(y_t)\right]
$$

- No se utiliza como criterio principal de selección.
- Se reporta con fines interpretativos.
- Conecta la predicción con la dirección esperada del movimiento.

**4. Coeficiente de determinación ($R^2$)**

- Se reporta como métrica descriptiva.
- No se utiliza como criterio principal debido a su limitada estabilidad en series financieras.


## **7.2. Enfoque `seq2seq`**

Dado que el problema consiste en la predicción de una secuencia futura completa, la comparación se realiza entre:

$$
\hat{Y}_{t+1:t+L} \quad \text{vs} \quad Y_{t+1:t+L}
$$

### **7.2.1 Métricas principales (criterio de selección)**

Las métricas se calculan de forma global concatenando todos los pasos de la secuencia en un único vector.

**1. MAE global**

- Error absoluto medio sobre todos los pasos y todas las muestras.

**2. RMSE global**

- Raíz del error cuadrático medio sobre todos los pasos y todas las muestras.

Estas métricas constituyen el criterio principal de comparación y ranking de modelos.


### **7.2.2 Métricas diagnósticas**


**3. Error por horizonte**

$$
MAE(k), \quad k = 1, \dots, L
$$

Permite:

- Observar la degradación del error a medida que aumenta el horizonte.
- Analizar la estabilidad temporal del modelo.

Este análisis es estrictamente diagnóstico y no se utiliza para selección final.



### **7.2.3 Métricas complementarias**


**4. Directional Accuracy (DA_last)**

Coincidencia de signo en el último paso de la secuencia:

$$
k = L
$$

- Conecta con la dirección esperada al horizonte final.
- Se utiliza solo con fines interpretativos.

**5. Coeficiente de determinación ($R^2$)**

- Se reporta como métrica descriptiva.
- No se utiliza como criterio principal de comparación.



## **7.3 Jerarquía de métricas**

**Métricas principales (criterio de selección)**
- MAE
- RMSE

**Métricas complementarias**
- Directional Accuracy (DA o DA_last)
- R²

**Métricas diagnósticas (solo seq2seq)**
- MAE(k)

La selección y descarte de modelos se realiza exclusivamente en base a las métricas principales evaluadas sobre VALID.


## **7.4 Función de cálculo de métricas**

### **7.4.1. Función de cálculo de métricas seq2one**

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

def compute_seq2one_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    *,
    compute_r2: bool = True,
    da_ignore_zeros: bool = True,
    allow_seq_inputs_take_last: bool = False,
) -> dict:
    """
    Calcula métricas simples y comparables para modelos seq2one.

    Parámetros
    ----------
    y_true : np.ndarray
        Valores reales con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    y_pred : np.ndarray
        Valores predichos con shape (n_samples,), (n_samples, 1)
        (opcional) (n_samples, seq_len) si allow_seq_inputs_take_last=True.
    compute_r2 : bool
        Si True, calcula R² sobre el vector completo.
    da_ignore_zeros : bool
        Si True, ignora casos donde el signo sea 0 en y_true o y_pred al calcular DA.
    allow_seq_inputs_take_last : bool
        Si True, permite inputs 2D (n_samples, seq_len) y toma el último paso [:, -1].
        Útil si algún modelo devuelve secuencia pero usted lo evalúa como many-to-one.

    Retorna
    -------
    metrics : dict
        Diccionario con métricas globales.
    """

    # 1) Convertir a np.ndarray y forzar float
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    # 2) Normalizar dimensiones hacia (n_samples,)
    def _to_1d(y: np.ndarray, name: str) -> np.ndarray:
        if y.ndim == 1:
            return y
        if y.ndim == 2:
            # (n, 1) -> (n,)
            if y.shape[1] == 1:
                return y.squeeze(1)
            # (n, seq_len) -> tomar último si se permite
            if allow_seq_inputs_take_last:
                return y[:, -1]
            raise ValueError(
                f"{name} con shape {y.shape} no es válido para seq2one. "
                f"Se esperaba (n_samples,) o (n_samples, 1)."
            )
        if y.ndim == 3 and y.shape[-1] == 1:
            # (n, seq_len, 1) -> (n, seq_len) y luego último si se permite
            y2 = y.squeeze(-1)
            if allow_seq_inputs_take_last:
                return y2[:, -1]
            raise ValueError(
                f"{name} con shape {y.shape} parece seq2seq. "
                f"Active allow_seq_inputs_take_last=True si quiere tomar el último paso."
            )
        raise ValueError(
            f"{name}.ndim={y.ndim} no es válido. "
            f"Se esperaba (n,), (n,1) o (n,seq_len) si allow_seq_inputs_take_last=True."
        )

    y_true = _to_1d(y_true, "y_true")
    y_pred = _to_1d(y_pred, "y_pred")

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"y_true y y_pred deben tener el mismo shape. "
            f"Recibido y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    n_samples = int(y_true.shape[0])
    if n_samples == 0:
        raise ValueError("y_true/y_pred no pueden estar vacíos.")

    # 3) Validación numérica básica
    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError("Se encontraron NaN o inf en y_true/y_pred. "
                         "Limpie o enmascare antes de calcular métricas.")

    # 4) Errores
    errors = y_pred - y_true
    abs_errors = np.abs(errors)

    # 5) Métricas principales
    mae = float(abs_errors.mean())
    rmse = float(np.sqrt((errors ** 2).mean()))

    # 6) Métrica direccional (DA)
    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if da_ignore_zeros:
        mask = (sign_true != 0) & (sign_pred != 0)
        da = float(np.mean(sign_true[mask] == sign_pred[mask])) if mask.any() else float("nan")
        da_n = int(mask.sum())
    else:
        da = float(np.mean(sign_true == sign_pred))
        da_n = n_samples

    metrics = {
        "MAE": mae,
        "RMSE": rmse,
        "DA": da,
        "DA_n": da_n,  # cuántas muestras realmente aportaron a DA (si ignore_zeros=True)
    }

    # 7) R² opcional
    if compute_r2:
        metrics["R2"] = float(r2_score(y_true, y_pred))

    return metrics

### **7.4.2. Función de cálculo de métricas seq2seq**

In [ ]:
import numpy as np
from sklearn.metrics import r2_score

def compute_seq2seq_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    compute_r2: bool = True,
    da_ignore_zeros: bool = True,
) -> dict:
    """
    Calcula métricas comparables para modelos seq2seq.

    Retorna:
    - Métricas globales (MAE, RMSE)
    - Métricas del último paso (MAE_last, RMSE_last)
    - DA_last
    - MAE por paso (diagnóstico)
    """

    # 1) Convertir a np.ndarray
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    # 2) Normalizar dimensiones a (n_samples, seq_len)
    if y_true.ndim == 3 and y_true.shape[-1] == 1:
        y_true = y_true.squeeze(-1)
    if y_pred.ndim == 3 and y_pred.shape[-1] == 1:
        y_pred = y_pred.squeeze(-1)

    if y_true.ndim != 2 or y_pred.ndim != 2:
        raise ValueError(
            f"Se espera shape (n_samples, seq_len) o (n_samples, seq_len, 1). "
            f"Recibido y_true.ndim={y_true.ndim}, y_pred.ndim={y_pred.ndim}"
        )

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"y_true y y_pred deben tener el mismo shape. "
            f"Recibido y_true={y_true.shape}, y_pred={y_pred.shape}"
        )

    n_samples, seq_len = y_true.shape
    if n_samples == 0 or seq_len == 0:
        raise ValueError("y_true/y_pred no pueden estar vacíos.")

    # 3) Validación numérica
    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError("Se encontraron NaN o inf en y_true/y_pred.")

    # 4) Errores
    errors = y_pred - y_true
    abs_errors = np.abs(errors)

    # ===============================
    # MÉTRICAS GLOBALES (seq2seq puro)
    # ===============================
    mae = float(abs_errors.mean())
    rmse = float(np.sqrt((errors ** 2).mean()))

    # ===============================
    # MÉTRICAS ÚLTIMO PASO (comparables con seq2one)
    # ===============================
    y_true_last = y_true[:, -1]
    y_pred_last = y_pred[:, -1]

    errors_last = y_pred_last - y_true_last

    mae_last = float(np.abs(errors_last).mean())
    rmse_last = float(np.sqrt((errors_last ** 2).mean()))

    # ===============================
    # Directional Accuracy (último paso)
    # ===============================
    sign_true = np.sign(y_true_last)
    sign_pred = np.sign(y_pred_last)

    if da_ignore_zeros:
        mask = (sign_true != 0) & (sign_pred != 0)
        da_last = float(np.mean(sign_true[mask] == sign_pred[mask])) if mask.any() else float("nan")
        da_last_n = int(mask.sum())
    else:
        da_last = float(np.mean(sign_true == sign_pred))
        da_last_n = int(n_samples)

    # ===============================
    # MAE por paso (diagnóstico)
    # ===============================
    mae_per_step = abs_errors.mean(axis=0)  # (seq_len,)

    metrics = {
        # Global seq2seq
        "MAE": mae,
        "RMSE": rmse,

        # Último paso (comparables con seq2one)
        "MAE_last": mae_last,
        "RMSE_last": rmse_last,

        # Dirección
        "DA_last": da_last,
        "DA_last_n": da_last_n,

        # Diagnóstico
        "MAE_per_step": mae_per_step.tolist(),
    }

    # ===============================
    # R² opcional
    # ===============================
    if compute_r2:
        metrics["R2"] = float(r2_score(y_true.ravel(), y_pred.ravel()))
        metrics["R2_last"] = float(r2_score(y_true_last, y_pred_last))

    return metrics

In [ ]:
#Ejemplo de uso:
#metrics = compute_seq2seq_metrics(y_true, y_pred)
#print(metrics["MAE"], metrics["RMSE"], metrics["DA_last"])